In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP02 - TP High Value
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Percentage of "high value" business arrangements/contracts maintained by the unit 
   during the assessment period.
   
   LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. HIGHEST VALUE FILTER: Cleans 'Contract_Amount' (removes $ and , symbols) and 
      casts to Double. Numerator identifies contracts >= 1,000,000.
   3. COMMA EXCEPTION HANDLING: Protects unit names containing commas (e.g., 'CAD PB - 
      RESL, Everyday Banking...') using a '[COMMA]' placeholder swap. This prevents 
      the SQL split/explode function from breaking a single unit name into multiple parts.
   4. FLATTEN LOBs: Uses LATERAL VIEW explode(split(...)) to expand comma-separated 
      LOB lists in Col K/L. Restores the '[COMMA]' to a real ',' after the split.
   5. SAFE MAPPING: Maps expanded LOBs to TPRM_Units using standard LIKE syntax 
      ('%' || String || '%') to avoid regex failures on special characters.
   6. DUAL-BRIDGE JOIN: Crucial fix for mixed-format mapping data. Bridges the 
      mapping table to the Master list on EITHER AU_Name OR Numeric BusinessID.
   7. AGGREGATING: Calculates Percentage (Numerator/Denominator * 100).
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- 2. Grab valid TPRM Mapping Strings
    -- Accommodates Mapping Table's mix of Numeric IDs and String Names
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    -- 3. Extract, Clean Amount, and Apply Comma Protection Dictionary
    SELECT 
        EngagementNumber,
        TRY_CAST(REPLACE(REPLACE(TRIM(Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount,
        
        -- EXCEPTION DICTIONARY: Chain replaces to shield internal commas from split()
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
            
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Flattened_LOBs AS (
    -- 4. Explode comma-separated strings and Restore the protected commas
    SELECT 
        EngagementNumber, 
        Cleaned_Amount,
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Third_Party_Risk_Data
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- 5. Map expanded rows to AU using Dual-Bridge and calculate Num/Denom
    SELECT 
        mast.BusinessID,
        -- Denominator: Total Distinct Engagements mapped to this AU
        COUNT(DISTINCT f.EngagementNumber) AS Denominator,
        -- Numerator: Distinct Engagements >= $1MM
        COUNT(DISTINCT CASE WHEN f.Cleaned_Amount >= 1000000 THEN f.EngagementNumber END) AS Numerator
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    
    -- DUAL-BRIDGE JOIN: Matches on either String Name OR Numeric BusinessID
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
        OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
        
    GROUP BY mast.BusinessID
)

-- 6. Final Output: Strictly structured per Master Anchor
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP02' AS QuestionID,               
    
    -- Final Percentage Math: Handles division by zero and appends '%'
    CASE 
        WHEN COALESCE(e.Denominator, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(e.Numerator AS DOUBLE) / e.Denominator) * 100, 2) AS STRING) || '%'
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP02 - AU Level Calculation Review
   PURPOSE: One row per AU showing denominator, numerator, bridge keys, and the
   exact percentage formula used for the final response.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    SELECT 
        EngagementNumber,
        TRY_CAST(REPLACE(REPLACE(TRIM(Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Flattened_LOBs AS (
    SELECT 
        EngagementNumber,
        Cleaned_Amount,
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Third_Party_Risk_Data
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),


TP_Names AS (
    -- Lookup: Distinct EngagementNumber -> ThirdPartyName
    SELECT DISTINCT
        TRIM(CAST(EngagementNumber AS STRING)) AS EngagementNumber,
        TRIM(ThirdPartyName) AS ThirdPartyName
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
      AND ThirdPartyName IS NOT NULL
),

Resolved_Bridge AS (
    SELECT 
        mast.BusinessID,
        mast.AU_Name,
        mast.Business_Segment,
        mast.In_ABAC_AU_List,
        f.EngagementNumber,
        f.Cleaned_Amount,
        f.Expanded_LOB,
        map.TPRM_Units,
        map.Mapping_ID_Or_Name,
        CASE WHEN f.Cleaned_Amount >= 1000000 THEN 1 ELSE 0 END AS Numerator_Flag,
        tp.ThirdPartyName
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
        OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
    LEFT JOIN TP_Names tp
        ON TRIM(CAST(f.EngagementNumber AS STRING)) = tp.EngagementNumber
),

AU_Debug AS (
    SELECT 
        BusinessID,
        MAX(AU_Name) AS AU_Name,
        MAX(Business_Segment) AS Business_Segment,
        MAX(In_ABAC_AU_List) AS In_ABAC_AU_List,
        COUNT(DISTINCT EngagementNumber) AS Denominator,
        COUNT(DISTINCT CASE WHEN Numerator_Flag = 1 THEN EngagementNumber END) AS Numerator,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(TPRM_Units))) AS Matched_TPRM_Units_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Mapping_ID_Or_Name))) AS Mapping_Bridge_Value_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Expanded_LOB))) AS Expanded_LOB_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(EngagementNumber))) AS Denominator_Engagement_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CASE WHEN Numerator_Flag = 1 THEN EngagementNumber END))) AS Numerator_Engagement_List,
        CONCAT_WS(', ', TRANSFORM(
            SORT_ARRAY(COLLECT_SET(CASE WHEN Numerator_Flag = 1 THEN NAMED_STRUCT('eng', CAST(EngagementNumber AS STRING), 'name', COALESCE(ThirdPartyName, '')) END)),
            x -> x.name
        )) AS Third_Party_Names,
        CONCAT_WS(', ', TRANSFORM(
            SORT_ARRAY(COLLECT_SET(CASE WHEN Numerator_Flag = 1 THEN NAMED_STRUCT('eng', CAST(EngagementNumber AS STRING), 'amt', COALESCE(CAST(Cleaned_Amount AS STRING), '')) END)),
            x -> x.amt
        )) AS Contract_Amounts
    FROM Resolved_Bridge
    GROUP BY BusinessID
)

-- Output columns:
-- BusinessID: Master AU ID from the ABAC AU anchor list.
-- AU_Name: Master AU name tied to BusinessID.
-- Business_Segment: Business segment carried from the master AU list.
-- QuestionID: Questionnaire code for this debug output.
-- Response: Final TP02 percentage shown to the customer.
-- Matched_TPRM_Units_List: TPRM unit values that matched this AU during bridging.
-- Mapping_Bridge_Value_List: Mapping-table assessable-unit values that bridged this AU.
-- Expanded_LOB_List: Flattened owning/using LOB values that participated in the bridge.
-- Denominator_Engagement_List: Distinct mapped engagements included in the denominator.
-- Numerator_Engagement_List: Distinct mapped engagements with cleaned amount at or above 1,000,000.
-- Third_Party_Names: Third party vendor names corresponding to each engagement in Numerator_Engagement_List, in matching positional order.
-- Contract_Amounts: Contract amount values corresponding to each engagement in Numerator_Engagement_List, in matching positional order.
-- Numerator: Final numerator engagement count.
-- Denominator: Final denominator engagement count.
-- Calculation_Formula: Plain-English explanation of the numerator/denominator percentage.
-- Debug_Reason: Short rule summary for how numerator and denominator are defined.
-- In_ABAC_AU_List: Confirms the AU came from the standard ABAC AU list.
SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'TP02' AS QuestionID,
    CASE 
        WHEN COALESCE(d.Denominator, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(d.Numerator AS DOUBLE) / d.Denominator) * 100, 2) AS STRING) || '%'
    END AS Response,
    COALESCE(d.Matched_TPRM_Units_List, 'n/a') AS Matched_TPRM_Units_List,
    COALESCE(d.Mapping_Bridge_Value_List, 'n/a') AS Mapping_Bridge_Value_List,
    COALESCE(d.Expanded_LOB_List, 'n/a') AS Expanded_LOB_List,
    COALESCE(d.Denominator_Engagement_List, 'n/a') AS Denominator_Engagement_List,
    COALESCE(d.Numerator_Engagement_List, 'n/a') AS Numerator_Engagement_List,
    COALESCE(d.Third_Party_Names, 'n/a') AS Third_Party_Names,
    COALESCE(d.Contract_Amounts, 'n/a') AS Contract_Amounts,
    COALESCE(d.Numerator, 0) AS Numerator,
    COALESCE(d.Denominator, 0) AS Denominator,
    CASE 
        WHEN COALESCE(d.Denominator, 0) = 0 THEN '0% because denominator = 0'
        ELSE CONCAT(
            CAST(d.Numerator AS STRING), ' / ', CAST(d.Denominator AS STRING), ' * 100 = ',
            CAST(ROUND((CAST(d.Numerator AS DOUBLE) / d.Denominator) * 100, 2) AS STRING), '%'
        )
    END AS Calculation_Formula,
    'Denominator counts distinct engagements mapped to the AU. Numerator counts distinct mapped engagements with cleaned amount >= 1000000.' AS Debug_Reason,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN AU_Debug d
    ON mast.BusinessID = d.BusinessID
ORDER BY mast.BusinessID;